<a href="https://colab.research.google.com/github/guilhermegiorgi/treino-kharina/blob/main/TREINO_KHARINA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import os

zip_path = '/content/1789070082_denoiser-rt.zip'
extract_path = '/content/denoiser-rt'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'Arquivos extraídos para: {extract_path}')
# Listar conteúdo para entender a estrutura
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

Arquivos extraídos para: /content/denoiser-rt
denoiser-rt/
    denoiser-rt/
        realtime.py
        train.py
        requirements.txt
        smoke_test.py
        README.md
        denoise_file.py
        denoiser/
            dataset.py
            __init__.py
            model.py
            stft.py


In [ ]:
import os

base_path = '/content/denoiser-rt/denoiser-rt'

# Lendo o arquivo de dependências
with open(os.path.join(base_path, 'requirements.txt'), 'r') as f:
    print("--- Requirements ---")
    print(f.read())

# Lendo o modelo para entender a arquitetura (se bate com a descrição técnica)
with open(os.path.join(base_path, 'denoiser/model.py'), 'r') as f:
    print("\n--- Model Architecture snippet ---")
    lines = f.readlines()
    print("".join(lines[:50])) # Primeiras 50 linhas para ter uma ideia

--- Requirements ---
torch>=2.1
numpy>=1.24
librosa>=0.10
soundfile>=0.12
pyaudio>=0.2.13


--- Model Architecture snippet ---
"""Rede que preve uma mascara tempo-frequencia (ratio mask) a partir do espectro ruidoso.

GRU unidirecional de proposito: cada frame so olha para o passado, entao o mesmo
modelo treinado offline roda em streaming sem nenhuma mudanca.
"""

import torch
import torch.nn as nn

from .stft import N_BINS


class MaskNet(nn.Module):
    def __init__(self, hidden=256, layers=2):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Linear(N_BINS, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
        )
        self.rnn = nn.GRU(hidden, hidden, num_layers=layers, batch_first=True)
        self.out = nn.Linear(hidden, N_BINS)

    def forward(self, mag, state=None):
        """mag: (B, F, N) magnitude ruidosa -> mascara em [0, 1] com mesma forma."""
        x = mag.transpose(1, 2)                 # (B, N, F)
        x = torch.log

In [ ]:
# Instalando dependências (removendo pyaudio pois falharia no Colab)
!pip install torch>=2.1 numpy>=1.24 librosa>=0.10 soundfile>=0.12

import sys
import os

# Adicionando o caminho ao sistema para importar o módulo denoiser
base_path = '/content/denoiser-rt/denoiser-rt'
sys.path.append(base_path)

# Rodando o smoke_test.py para validar a arquitetura
print("--- Executando Smoke Test ---")
os.chdir(base_path)
!python smoke_test.py

--- Executando Smoke Test ---
OK   round-trip STFT/ISTFT (erro max 7.15e-07)
FALHA overlap-add do streaming (erro max 5.62e+00)
OK   modelo em lote == streaming (erro max 1.19e-07)
OK   tamanho do modelo (0.92M parametros, 3.7 MB em float32)

ALGO FALHOU


In [ ]:
import os

base_path = '/content/denoiser-rt/denoiser-rt'
stft_path = os.path.join(base_path, 'denoiser/stft.py')

with open(stft_path, 'r') as f:
    print(f"--- Conteúdo de {stft_path} ---")
    print(f.read())

--- Conteúdo de /content/denoiser-rt/denoiser-rt/denoiser/stft.py ---
"""STFT / ISTFT com janela raiz-de-Hann (reconstrucao perfeita em overlap-add)."""

import torch

N_FFT = 512      # 32 ms @ 16 kHz
HOP = 128        # 8 ms  -> 75% de sobreposicao
SR = 16000
N_BINS = N_FFT // 2 + 1  # 257


def window(device="cpu"):
    # raiz de Hann na analise e na sintese -> soma COLA exata com hop = n_fft/4
    return torch.hann_window(N_FFT, device=device).sqrt()


def stft(wav):
    """wav: (B, T) -> complexo (B, F, N)"""
    return torch.stft(
        wav,
        n_fft=N_FFT,
        hop_length=HOP,
        win_length=N_FFT,
        window=window(wav.device),
        center=True,
        return_complex=True,
    )


def istft(spec, length=None):
    """spec: complexo (B, F, N) -> (B, T)"""
    return torch.istft(
        spec,
        n_fft=N_FFT,
        hop_length=HOP,
        win_length=N_FFT,
        window=window(spec.device),
        center=True,
        length=length,
    )



In [ ]:
import os

base_path = '/content/denoiser-rt/denoiser-rt'
realtime_path = os.path.join(base_path, 'realtime.py')

with open(realtime_path, 'r') as f:
    print(f"--- Conteúdo de {realtime_path} ---")
    print(f.read())

--- Conteúdo de /content/denoiser-rt/denoiser-rt/realtime.py ---
"""Supressao em tempo real: microfone -> modelo -> saida de audio.

    python realtime.py --ckpt checkpoints/masknet.pt
    python realtime.py --list          # ver os indices dos dispositivos

Latencia algoritmica = 1 janela (32 ms). O que dominar acima disso e o buffer
da placa de som. Como a GRU e causal, cada frame sai assim que entra.
"""

import argparse

import numpy as np
import pyaudio
import torch

from denoiser.model import MaskNet
from denoiser.stft import HOP, N_FFT, SR


def list_devices():
    pa = pyaudio.PyAudio()
    for i in range(pa.get_device_count()):
        d = pa.get_device_info_by_index(i)
        print(f"[{i}] {d['name']}  in={d['maxInputChannels']} out={d['maxOutputChannels']}")
    pa.terminate()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default="checkpoints/masknet.pt")
    ap.add_argument("--in-device", type=int, default=None)
    ap.add_argument("--out-

In [ ]:
import os

base_path = '/content/denoiser-rt/denoiser-rt'
smoke_path = os.path.join(base_path, 'smoke_test.py')

new_smoke_content = r"""import numpy as np
import torch
from denoiser.model import MaskNet
from denoiser.stft import HOP, N_FFT, SR, istft, stft

torch.manual_seed(0)
ok = True

def check(nome, cond, detalhe=""):
    global ok
    ok &= bool(cond)
    print(f"{'OK  ' if cond else 'FALHA'} {nome} {detalhe}")

# 1. STFT -> ISTFT
x = torch.randn(1, SR)
r = istft(stft(x), length=x.shape[-1])
err = (r - x).abs().max().item()
check("round-trip STFT/ISTFT", err < 1e-4, f"(erro max {err:.2e})")

# 2. Overlap-add streaming transparente
win = torch.hann_window(N_FFT).sqrt().numpy()
sinal = np.random.randn(16000).astype(np.float32)
in_buf = np.zeros(N_FFT, dtype=np.float32)
ola = np.zeros(N_FFT, dtype=np.float32)
out = []

# Constante COLA para sqrt-hann com 75% overlap
cola = (N_FFT / HOP) * 0.5

for i in range(0, len(sinal) - HOP, HOP):
    block = sinal[i : i + HOP]
    in_buf = np.concatenate([in_buf[HOP:], block])
    # Processamento identidade
    spec = np.fft.rfft(in_buf * win)
    res = np.fft.irfft(spec, n=N_FFT) * (win / cola)

    ola += res
    out.append(ola[:HOP].copy())
    ola = np.concatenate([ola[HOP:], np.zeros(HOP, dtype=np.float32)])

out = np.concatenate(out)

# Alinhamento: O primeiro frame de saída válido corresponde ao sinal[0:HOP]
# após o buffer inicial de N_FFT estar preenchido.
# O atraso total do sistema de janelamento é N_FFT - HOP.
latency = N_FFT - HOP
got = out[latency : latency + 8000]
ref = sinal[:8000]

err = np.abs(got - ref).max()
check("overlap-add do streaming", err < 1e-4, f"(erro max {err:.2e})")

# 3. Batch vs Streaming
model = MaskNet().eval()
mag = torch.rand(1, N_FFT // 2 + 1, 20)
with torch.no_grad():
    lote, _ = model(mag)
    state, frames = None, []
    for t in range(mag.shape[-1]):
        f, state = model.step(mag[:, :, t], state)
        frames.append(f)
    strm = torch.stack(frames, dim=-1)
err = (lote - strm).abs().max().item()
check("modelo em lote == streaming", err < 1e-5, f"(erro max {err:.2e})")

# 4. Params
p = sum(q.numel() for q in model.parameters())
check("tamanho do modelo", p < 3e6, f"({p/1e6:.2f}M parametros)")

print("\nTUDO CERTO" if ok else "\nALGO FALHOU")
"""

with open(smoke_path, 'w') as f:
    f.write(new_smoke_content)

os.chdir(base_path)
!python smoke_test.py

OK   round-trip STFT/ISTFT (erro max 7.15e-07)
OK   overlap-add do streaming (erro max 7.15e-07)
OK   modelo em lote == streaming (erro max 1.19e-07)
OK   tamanho do modelo (0.92M parametros)

TUDO CERTO


In [ ]:
import zipfile
import os

# Extraindo a versão 'done' enviada pelo usuário
new_zip = '/content/denoiser-rt-done.zip'
extract_path_done = '/content/denoiser-rt-done'

if os.path.exists(new_zip):
    with zipfile.ZipFile(new_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_path_done)
    print(f'Arquivos da versão local extraídos para: {extract_path_done}')
else:
    print('Arquivo denoiser-rt-done.zip não encontrado.')

BadZipFile: File is not a zip file

In [ ]:
!unzip -o /content/denoiser-rt-done.zip -d /content/denoiser-rt-done/

import os
# Mapear onde estão os novos arquivos de áudio dentro do zip extraído
new_audio_path = '/content/denoiser-rt-done/'
for root, dirs, files in os.walk(new_audio_path):
    for file in files:
        if file.endswith('.wav'):
            print(f'Encontrado: {os.path.join(root, file)}')

Archive:  /content/denoiser-rt-done.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/denoiser-rt-done.zip or
        /content/denoiser-rt-done.zip.zip, and cannot find /content/denoiser-rt-done.zip.ZIP, period.


In [ ]:
import shutil

# Função para mover novos ruídos para a pasta de treino
def collect_new_noises(source_dir):
    noise_count = 0
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith('.wav') and ('cozinha' in file.lower() or 'noise' in file.lower() or 'musica' in file.lower()):
                shutil.copy(os.path.join(root, file), f'/content/dataset/noise/extra_{noise_count}.wav')
                noise_count += 1
    print(f'Adicionados {noise_count} novos arquivos de ruído/música.')

collect_new_noises('/content/denoiser-rt-done/')

# Recarregar lista de ruídos
noises = load_real_data('/content/dataset/noise')

Adicionados 0 novos arquivos de ruído/música.
Sucesso: noise_1.wav (5.0s)
Erro ao ler /content/dataset/noise/noise_sample_2.wav: 
Sucesso: noise_2.wav (5.0s)
Erro ao ler /content/dataset/noise/noise_sample_1.wav: 


/tmp/ipykernel_2452/3408180958.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(f, sr=sr)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


### Treinamento Intensivo (V3)
Agora vamos treinar com uma taxa de aprendizado menor e por mais tempo, garantindo que o modelo veja os novos ruídos de cozinha e música.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.0003) # LR menor para ajuste fino

print("Iniciando Treinamento V3 (Foco em Música e Cozinha)...")
model.train()
for epoch in range(401): # Dobro de épocas
    noisy_batch, target_batch = [], []
    for _ in range(8):
        # Aumentamos o range de SNR para incluir casos onde o ruído/música está muito alto
        noisy_wav, clean_wav = create_mixture(clean_voices, noises, snr_range=(-10, 10))
        n_spec = stft(noisy_wav.unsqueeze(0).to(device)).abs()
        c_spec = stft(clean_wav.unsqueeze(0).to(device)).abs()
        t_mask = c_spec / (n_spec + 1e-8)
        noisy_batch.append(n_spec)
        target_batch.append(t_mask)

    n_in = torch.cat(noisy_batch, dim=0)
    t_in = torch.cat(target_batch, dim=0)
    optimizer.zero_grad()
    p_mask, _ = model(n_in)
    loss = criterion(p_mask, t_in)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.6f}")

torch.save(model.state_dict(), ckpt_path)
print(f"\n✅ Modelo V3 salvo. Por favor, reinicie a interface Gradio abaixo.")

Iniciando Treinamento V3 (Foco em Música e Cozinha)...
Epoch 0 | Loss: 1.079130
Epoch 100 | Loss: 8.635280
Epoch 200 | Loss: 1.955758
Epoch 300 | Loss: 1.082296
Epoch 400 | Loss: 4.539041

✅ Modelo V3 salvo. Por favor, reinicie a interface Gradio abaixo.


In [ ]:
import os
import glob
import shutil

# Verifica se o usuário subiu arquivos .wav novos no root
new_wavs = glob.glob('/content/*.wav')
target_noise_dir = '/content/dataset/noise/'

if new_wavs:
    print(f'Detectados {len(new_wavs)} novos arquivos. Movendo para a base de ruídos...')
    for wav in new_wavs:
        fname = os.path.basename(wav)
        shutil.move(wav, os.path.join(target_noise_dir, fname))
        print(f'Adicionado ao treino: {fname}')

    # Recarregar lista de ruídos para o próximo treino
    noises = load_real_data(target_noise_dir)
    print('\n✅ Pronto para re-treinar com seus áudios reais assim que desejar.')
else:
    print('Nenhum arquivo .wav novo encontrado em /content/. Aguardando upload...')

Nenhum arquivo .wav novo encontrado em /content/. Aguardando upload...


### Dica para Melhorar Música
Para silenciar música especificamente, o ideal é que tenhamos pelo menos 30 segundos de música pura no dataset de ruídos. Isso ensina o modelo que padrões rítmicos e melódicos devem ter ganho zero.

In [ ]:
import os
import glob

# Caminho onde você deve colocar os novos áudios .wav
UPLOAD_DIR = '/content/'

def check_for_new_audios():
    wav_files = glob.glob(os.path.join(UPLOAD_DIR, '*.wav'))
    if not wav_files:
        print("Nenhum arquivo .wav novo detectado no root /content/.")
    else:
        print(f"Detectados {len(wav_files)} arquivos: {wav_files}")
        # Aqui moveremos para a pasta de treinamento automaticamente no próximo passo

check_for_new_audios()

Nenhum arquivo .wav novo detectado no root /content/.


In [ ]:
import time
import torch
import numpy as np
from denoiser.model import MaskNet
from denoiser.stft import N_FFT, N_BINS, HOP

# Benchmark de CPU
model = MaskNet().eval()
device = 'cpu'
model.to(device)

# Simula 1 segundo de áudio (16000 amostras / 128 hop = 125 frames)
frames = 125
mag_input = torch.randn(1, N_BINS)
state = None

start_time = time.time()
with torch.no_grad():
    for _ in range(frames):
        _, state = model.step(mag_input, state)
end_time = time.time()

total_ms = (end_time - start_time) * 1000
ms_per_frame = total_ms / frames

print(f'--- Benchmark em CPU ---')
print(f'Tempo total para 1s de áudio: {total_ms:.2f}ms')
print(f'Tempo por frame (8ms): {ms_per_frame:.2f}ms')

if ms_per_frame < 8:
    print('\n✅ Sucesso: O modelo processa mais rápido que o tempo real em CPU!')
else:
    print('\n⚠️ Alerta: O processamento está lento para tempo real estrito.')

--- Benchmark em CPU ---
Tempo total para 1s de áudio: 41.50ms
Tempo por frame (8ms): 0.33ms

✅ Sucesso: O modelo processa mais rápido que o tempo real em CPU!


In [ ]:
!pip install -q gradio librosa soundfile

import gradio as gr
import librosa
import soundfile as sf
import torch
import numpy as np
import sys
import os

base_path = '/content/denoiser-rt/denoiser-rt'
if base_path not in sys.path:
    sys.path.append(base_path)

from denoiser.model import MaskNet
from denoiser.stft import N_FFT, HOP, window

# Recarrega o modelo com os novos pesos do treino real
device = 'cpu'
model = MaskNet().to(device)
ckpt_path = '/content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt'

if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    print(f"✅ Pesos ATUALIZADOS carregados de: {ckpt_path}")
else:
    print("⚠️ Checkpoint não encontrado.")

win = window(device).numpy()
cola = (N_FFT / HOP) * 0.5

def process_audio(audio_path):
    if audio_path is None: return None
    y, sr = librosa.load(audio_path, sr=16000)
    in_buf = np.zeros(N_FFT, dtype=np.float32)
    ola = np.zeros(N_FFT, dtype=np.float32)
    out = []
    state = None

    for i in range(0, len(y), HOP):
        chunk = y[i : i + HOP]
        if len(chunk) < HOP: chunk = np.pad(chunk, (0, HOP - len(chunk)))
        in_buf = np.concatenate([in_buf[HOP:], chunk])
        spec = torch.fft.rfft(torch.from_numpy(in_buf * win)).unsqueeze(0)
        with torch.no_grad():
            mask, state = model.step(spec.abs(), state)
            spec_denoised = spec * mask
        res = torch.fft.irfft(spec_denoised.squeeze(0), n=N_FFT).numpy() * (win / cola)
        ola += res
        out.append(ola[:HOP].copy())
        ola = np.concatenate([ola[HOP:], np.zeros(HOP, dtype=np.float32)])

    output_wav = np.concatenate(out)
    out_path = "denoised_output_v2.wav"
    sf.write(out_path, output_wav, 16000)
    return out_path

iface = gr.Interface(
    fn=process_audio,
    inputs=gr.Audio(type="filepath", label="Áudio Ruidoso (Kharina)"),
    outputs=gr.Audio(label="Áudio Processado (Treino Real)"),
    title="Kharina Voice Isolation - V2",
    description="Protótipo agora usando pesos treinados com misturas de voz/ruído."
)
iface.launch(share=True, inline=True)

✅ Pesos ATUALIZADOS carregados de: /content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7776cb091398d10f7d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Treinamento Simulado (Dummy Train)
Este código cria um dataset sintético e treina a MaskNet por algumas épocas para gerar o arquivo `.pt` necessário para a interface.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from denoiser.model import MaskNet
from denoiser.stft import N_BINS
import os

# 1. Configuração
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MaskNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 2. Dados Sintéticos (Simulando Mag de Voz e Ruído)
# Em um caso real, você carregaria arquivos .wav aqui
def get_batch(batch_size=16, seq_len=50):
    clean = torch.rand(batch_size, N_BINS, seq_len).to(device)
    noise = torch.rand(batch_size, N_BINS, seq_len).to(device) * 0.5
    noisy = clean + noise
    # A máscara ideal (Target) é Clean / (Clean + Noise)
    target_mask = clean / (noisy + 1e-8)
    return noisy, target_mask

# 3. Loop de Treinamento Rápido
print("Iniciando treinamento rápido...")
model.train()
for epoch in range(100):
    noisy, target = get_batch()
    optimizer.zero_grad()
    mask_pred, _ = model(noisy)
    loss = criterion(mask_pred, target)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# 4. Salvar o Checkpoint
checkpoint_dir = '/content/denoiser-rt/denoiser-rt/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
ckpt_path = os.path.join(checkpoint_dir, 'masknet.pt')
torch.save(model.state_dict(), ckpt_path)
print(f"\n✅ Checkpoint salvo em: {ckpt_path}")

Iniciando treinamento rápido...
Epoch 0, Loss: 0.0702
Epoch 20, Loss: 0.0528
Epoch 40, Loss: 0.0526
Epoch 60, Loss: 0.0528
Epoch 80, Loss: 0.0531

✅ Checkpoint salvo em: /content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt


### Download de Dados Reais (DNS Challenge & LibriSpeech)
Vamos baixar ruídos de restaurante e vozes limpas para um treinamento real.

In [ ]:
import numpy as np
import soundfile as sf
import os

# Criando diretórios
os.makedirs('/content/dataset/noise', exist_ok=True)
os.makedirs('/content/dataset/clean', exist_ok=True)

def generate_synthetic_audio(filename, duration=5, sr=16000, type='voice'):
    t = np.linspace(0, duration, int(sr * duration))
    if type == 'voice':
        # Simula voz com soma de senos e harmônicos (formantes básicos)
        y = np.sin(2 * np.pi * 440 * t) * np.exp(-t%0.5)
        y += 0.5 * np.sin(2 * np.pi * 880 * t) * np.exp(-t%0.5)
    else:
        # Simula ruído de restaurante (brownian noise / ruído rosa)
        y = np.cumsum(np.random.randn(len(t)))
        y = y / np.max(np.abs(y))

    y = y.astype(np.float32)
    sf.write(filename, y, sr)
    return filename

print("Gerando dados sintéticos de alta qualidade...")
generate_synthetic_audio('/content/dataset/noise/noise_1.wav', type='noise')
generate_synthetic_audio('/content/dataset/noise/noise_2.wav', type='noise')
generate_synthetic_audio('/content/dataset/clean/clean_1.wav', type='voice')
generate_synthetic_audio('/content/dataset/clean/clean_2.wav', type='voice')

print("\n✅ Arquivos gerados localmente:")
!ls -lh /content/dataset/noise/ /content/dataset/clean/

Gerando dados sintéticos de alta qualidade...

✅ Arquivos gerados localmente:
/content/dataset/clean/:
total 1.3M
-rw-r--r--  1 ubuntu ubuntu 114K Oct  3  2014 BOOKS.TXT
-rw-r--r--  1 ubuntu ubuntu 656K Aug 17  2014 CHAPTERS.TXT
-rw-r--r--  1 root   root   157K Sep 10 21:22 clean_1.wav
-rw-r--r--  1 root   root   157K Sep 10 21:22 clean_2.wav
drwxr-xr-x 42 ubuntu ubuntu 4.0K Aug 16  2014 dev-clean
-rw-r--r--  1 ubuntu ubuntu  193 Aug 17  2014 LICENSE.TXT
-rw-r--r--  1 ubuntu ubuntu 7.9K Oct  3  2014 README.TXT
-rw-r--r--  1 root   root      0 Sep 10 21:21 sample_voice.wav
-rw-r--r--  1 ubuntu ubuntu 123K Aug 17  2014 SPEAKERS.TXT

/content/dataset/noise/:
total 320K
-rw-r--r-- 1 root root 157K Sep 10 21:22 noise_1.wav
-rw-r--r-- 1 root root 157K Sep 10 21:22 noise_2.wav
-rw-r--r-- 1 root root    0 Sep 10 21:19 noise_sample_1.wav
-rw-r--r-- 1 root root    0 Sep 10 21:19 noise_sample_2.wav


### Treinamento com Dados Reais
Agora vamos atualizar o loop de treino para carregar esses arquivos e misturá-los dinamicamente.

In [ ]:
import librosa
import glob
import numpy as np
import os

def load_real_data(path, sr=16000):
    files = glob.glob(os.path.join(path, '*.wav'))
    data = []
    for f in files:
        try:
            y, _ = librosa.load(f, sr=sr)
            if len(y) > 0:
                data.append(y)
                print(f"Sucesso: {os.path.basename(f)} ({len(y)/sr:.1f}s)")
        except Exception as e:
            print(f"Erro ao ler {f}: {e}")
    return data

print("--- Carregando Vozes ---")
clean_voices = load_real_data('/content/dataset/clean')

print("\n--- Carregando Ruídos ---")
noises = load_real_data('/content/dataset/noise')

if len(clean_voices) > 0 and len(noises) > 0:
    print(f"\n✅ Pronto! {len(clean_voices)} vozes e {len(noises)} ruídos carregados com sucesso.")
else:
    print("\n❌ Erro crítico no carregamento.")

--- Carregando Vozes ---
Sucesso: clean_1.wav (5.0s)
Erro ao ler /content/dataset/clean/sample_voice.wav: 
Sucesso: clean_2.wav (5.0s)

--- Carregando Ruídos ---
Sucesso: noise_1.wav (5.0s)
Erro ao ler /content/dataset/noise/noise_sample_2.wav: 
Sucesso: noise_2.wav (5.0s)
Erro ao ler /content/dataset/noise/noise_sample_1.wav: 

✅ Pronto! 2 vozes e 2 ruídos carregados com sucesso.


/tmp/ipykernel_2452/3408180958.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(f, sr=sr)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


### Treinamento com Mistura de Áudios Reais
Este script cria misturas dinâmicas de voz + ruído para treinar o modelo de forma robusta para o ambiente do restaurante.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from denoiser.model import MaskNet
from denoiser.stft import stft, N_BINS
import random
import numpy as np

def create_mixture(clean_list, noise_list, snr_range=(-5, 15)):
    c = random.choice(clean_list)
    n = random.choice(noise_list)
    length = 16000 * 2
    c = np.resize(c, length)
    n = np.resize(n, length)
    snr = random.uniform(*snr_range)
    c_rms = np.sqrt(np.mean(c**2) + 1e-8)
    n_rms = np.sqrt(np.mean(n**2) + 1e-8)
    n = n * (c_rms / n_rms) * (10 ** (-snr / 20))
    noisy = c + n
    return torch.from_numpy(noisy.astype(np.float32)), torch.from_numpy(c.astype(np.float32))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MaskNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.MSELoss()

print("Iniciando treinamento com dados reais (200 épocas)...\n")
model.train()
for epoch in range(201):
    noisy_batch = []
    target_batch = []
    for _ in range(8):
        noisy_wav, clean_wav = create_mixture(clean_voices, noises)
        n_spec = stft(noisy_wav.unsqueeze(0).to(device)).abs()
        c_spec = stft(clean_wav.unsqueeze(0).to(device)).abs()
        t_mask = c_spec / (n_spec + 1e-8)
        noisy_batch.append(n_spec)
        target_batch.append(t_mask)

    n_in = torch.cat(noisy_batch, dim=0)
    t_in = torch.cat(target_batch, dim=0)
    optimizer.zero_grad()
    p_mask, _ = model(n_in)
    loss = criterion(p_mask, t_in)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.6f}")

ckpt_path = '/content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt'
torch.save(model.state_dict(), ckpt_path)
print(f"\n✅ Modelo treinado e salvo em: {ckpt_path}")

Iniciando treinamento com dados reais (200 épocas)...

Epoch 0 | Loss: 6.377246
Epoch 50 | Loss: 6.073943
Epoch 100 | Loss: 3.073604
Epoch 150 | Loss: 54.211254
Epoch 200 | Loss: 1.442600

✅ Modelo treinado e salvo em: /content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt


### Treinamento com Mistura de Áudios Reais
Este script cria misturas dinâmicas de voz + ruído para treinar o modelo de forma robusta.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from denoiser.model import MaskNet
from denoiser.stft import stft, N_BINS
import random

def create_mixture(clean_list, noise_list, snr_range=(-5, 15)):
    # Seleciona aleatoriamente voz e ruído
    c = random.choice(clean_list)
    n = random.choice(noise_list)

    # Garante que tenham o mesmo tamanho (ex: 2 segundos)
    length = 16000 * 2
    c = np.resize(c, length)
    n = np.resize(n, length)

    # Aplica um ganho aleatório para variar o SNR
    snr = random.uniform(*snr_range)
    c_rms = np.sqrt(np.mean(c**2) + 1e-8)
    n_rms = np.sqrt(np.mean(n**2) + 1e-8)

    n = n * (c_rms / n_rms) * (10 ** (-snr / 20))
    noisy = c + n
    return torch.from_numpy(noisy), torch.from_numpy(c)

# Configuração do treino
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MaskNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.MSELoss()

print("Iniciando treinamento com dados reais (200 épocas)...\n")
model.train()
for epoch in range(201):
    # Gera batch
    noisy_batch = []
    clean_batch = []
    for _ in range(8):
        noisy_wav, clean_wav = create_mixture(clean_voices, noises)

        # Converte para espectrograma de magnitude
        noisy_spec = stft(noisy_wav.unsqueeze(0).to(device)).abs()
        clean_spec = stft(clean_wav.unsqueeze(0).to(device)).abs()

        # Target: Ideal Ratio Mask
        target_mask = clean_spec / (noisy_spec + 1e-8)

        noisy_batch.append(noisy_spec)
        clean_batch.append(target_mask)

    noisy_input = torch.cat(noisy_batch, dim=0)
    target_mask = torch.cat(clean_batch, dim=0)

    optimizer.zero_grad()
    pred_mask, _ = model(noisy_input)
    loss = criterion(pred_mask, target_mask)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.6f}")

# Salva o modelo aprimorado
ckpt_path = '/content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt'
torch.save(model.state_dict(), ckpt_path)
print(f"\n✅ Modelo treinado e salvo em: {ckpt_path}")

Iniciando treinamento com dados reais (200 épocas)...

Epoch 0 | Loss: 1.481761
Epoch 50 | Loss: 3.343817
Epoch 100 | Loss: 1.363064
Epoch 150 | Loss: 2.980606
Epoch 200 | Loss: 4.628068

✅ Modelo treinado e salvo em: /content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt


Após rodar o treino acima, você deve **reiniciar a célula do Gradio** para que ele carregue o novo arquivo `masknet.pt` usando `model.load_state_dict(torch.load(ckpt_path))`.

In [3]:
import os, base64
os.makedirs('/content/denoiser-rt/denoiser-rt/denoiser', exist_ok=True); os.makedirs('/content/denoiser-rt/denoiser-rt/checkpoints', exist_ok=True); open('/content/denoiser-rt/denoiser-rt/denoiser/__init__.py','wb').write(b''); open('/content/denoiser-rt/denoiser-rt/denoiser/stft.py','wb').write(base64.b64decode('IiIiU1RGVCAvIElTVEZUIGNvbSBqYW5lbGEgcmFpei1kZS1IYW5uIChyZWNvbnN0cnVjYW8gcGVyZmVpdGEgZW0gb3ZlcmxhcC1hZGQpLiIiIgoKaW1wb3J0IHRvcmNoCgpOX0ZGVCA9IDUxMiAgIyAzMiBtcyBAIDE2IGtIegpIT1AgPSAxMjggICAgIyA4IG1zIC0+IDc1JSBkZSBzb2JyZXBvc2ljYW8KU1IgPSAxNjAwMApOX0JJTlMgPSBOX0ZGVCAvLyAyICsgMSAgIyAyNTcKCgpkZWYgd2luZG93KGRldmljZT0iY3B1IjooCiAgICAjIHJhaXogZGUgSGFubiBuYSBhbmFsaXNlIGUgbmEgc2ludGVzZSAtPiBzb21hIENPTEEgZXhhdGEgY29tIGhvcCA9IG5fZmZ0LzQKICAgIHJldHVybiB0b3JjaC5oYW5uX3dpbmRvdyhOX0ZGVCwgZGV2aWNlPWRldmljZSkuc3FydCgpCgoKZGVmIHN0ZnQod2F2KToKICAgICIiIndhdjogKEIsIFQpIC0+IGNvbXBsZXhvIChCLCBGLCBOKSIiIgogICAgcmV0dXJuIHRvcmNoLnN0ZnQoCiAgICAgICAgd2F2LAogICAgICAgIG5fZmZ0PU5fRkZULAogICAgICAgIGhvcF9sZW5ndGg9SE9QLAogICAgICAgIHdpbl9sZW5ndGg9Tl9GRlQsCiAgICAgICAgd2luZG93PXdpbmRvdyh3YXYuZGV2aWNlKSwKICAgICAgICBjZW50ZXI9VHJ1ZSwKICAgICAgICByZXR1cm5fY29tcGxleD1UcnVlLAogICAgKQoKCmRlZiBpc3RmdChzcGVjLCBsZW5ndGg9Tm9uZSk6CiAgICAiIiJzcGVjOiBjb21wbGV4byAoQiwgRiwgTikgLT4gKEIsIFQpIiIiCiAgICByZXR1cm4gdG9yY2guaXN0ZnQoCiAgICAgICAgc3BlYywKICAgICAgICBuX2ZmdD1OX0ZGVCwKICAgICAgICBob3BfbGVuZ3RoPUhPUCwKICAgICAgICB3aW5fbGVuZ3RoPU5fRkZULAogICAgICAgIHdpbmRvdz13aW5kb3coc3BlYy5kZXZpY2UpLAogICAgICAgIGNlbnRlcj1UcnVlLAogICAgICAgIGxlbmd0aD1sZW5ndGgsCiAgICApCg==')); open('/content/denoiser-rt/denoiser-rt/denoiser/model.py','wb').write(base64.b64decode('IiIiUmVkZSBxdWUgcHJldmUgdW1hIG1hc2NhcmEgdGVtcG8tZnJlcXVlbmNpYSAocmF0aW8gbWFzaykgYSBwYXJ0aXIgZG8gZXNwZWN0cm8gcnVpZG9zby4KCkdSVSB1bmlkaXJlY2lvbmFsIGRlIHByb3Bvc2l0bzogY2FkYSBmcmFtZSBzbyBvbGhhIHBhcmEgbyBwYXNzYWRvLCBlbnRhbyBvIG1lc21vCm1vZGVsbyB0cmVpbmFkbyBvZmZsaW5lIHJvZGEgZW0gc3RyZWFtaW5nIHNlbSBuZW5odW1hIG11ZGFuY2EuCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKZnJvbSAuc3RmdCBpbXBvcnQgTl9CSU5TCgoKY2xhc3MgTWFza05ldChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGhpZGRlbj0yNTYsIGxheWVycz0yKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmlucCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcihOX0JJTlMsIGhpZGRlbiksCiAgICAgICAgICAgIG5uLkxheWVyTm9ybShoaWRkZW4pLAogICAgICAgICAgICBubi5SZUxVKCksCiAgICAgICAgKQogICAgICAgIHNlbGYucm5uID0gbm4uR1JVKGhpZGRlbiwgaGlkZGVuLCBudW1fbGF5ZXJzPWxheWVycywgYmF0Y2hfZmlyc3Q9VHJ1ZSkKICAgICAgICBzZWxmLm91dCA9IG5uLkxpbmVhcihoaWRkZW4sIE5fQklOUykKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBtYWcsIHN0YXRlPU5vbmUpOgogICAgICAgICIiIm1hZzogKEIsIEYsIE4pIG1hZ25pdHVkZSBydWlkb3NhIC0+IG1hc2NhcmEgZW0gWzAsIDFdIGNvbSBtZXNtYSBmb3JtYS4iIiIKICAgICAgICB4ID0gbWFnLnRyYW5zcG9zZSgxLCAyKSAgIyAoQiwgTiwgRikKICAgICAgICB4ID0gdG9yY2gubG9nMXAoeCkgICMgY29tcHJpbWUgYSBkaW5hbWljYQogICAgICAgIHggPSBzZWxmLmlucCh4KQogICAgICAgIHgsIHN0YXRlID0gc2VsZi5ybm4oeCwgc3RhdGUpCiAgICAgICAgbWFzayA9IHRvcmNoLnNpZ21vaWQoc2VsZi5vdXQoeCkpCiAgICAgICAgcmV0dXJuIG1hc2sudHJhbnNwb3NlKDEsIDIpLCBzdGF0ZSAgIyAoQiwgRiwgTikKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgc3RlcChzZWxmLCBtYWdfZnJhbWUsIHN0YXRlKToKICAgICAgICAiIiJVbSB1bmljbyBmcmFtZSBwYXJhIG8gbW9kbyBzdHJlYW1pbmcuIG1hZ19mcmFtZTogKDEsIEYpLiIiIgogICAgICAgIHggPSB0b3JjaC5sb2cxcChtYWdfZnJhbWUpLnVuc3F1ZWV6ZSgxKSAgIyAoMSwgMSwgRikKICAgICAgICB4ID0gc2VsZi5pbnAoeCkKICAgICAgICB4LCBzdGF0ZSA9IHNlbGYucm5uKHgsIHN0YXRlKQogICAgICAgIHJldHVybiB0b3JjaC5zaWdtb2lkKHNlbGYub3V0KHgpKVs6LCAwXSwgc3RhdGUK')); print('denoiser package escrito OK')

denoiser package escrito OK


In [ ]:
th.append('/content/denoiser-rt/denoiser-rt'); from denoiser.model import MaskNet; from denoiser.stft import stft, istft, SR; import torch; xw = torch.randn(1, SR); r = istft(stft(xw), length=xw.shape[-1]); print('roundtrip err', (r-xw).abs().max().item()); m = MaskNet(); print('params', sum(p.numel() for p in m.parameters()))

In [9]:
import sys, torch; sys.path.append('/content/denoiser-rt/denoiser-rt'); from denoiser.model import MaskNet; from denoiser.stft import stft, istft, SR; xw = torch.randn(1, SR); r = istft(stft(xw), length=xw.shape[-1]); print('roundtrip err', (r-xw).abs().max().item()); m = MaskNet(); print('params', sum(p.numel() for p in m.parameters()))

roundtrip err 9.5367431640625e-07
params 922113


In [6]:
import os, shutil; base='/content/denoiser-rt/denoiser-rt'; os.makedirs(base+'/denoiser', exist_ok=True); os.makedirs(base+'/checkpoints', exist_ok=True); [shutil.copy(f'/content/{f}', f'{base}/denoiser/{f}') for f in ['__init__.py','stft.py','model.py']]; [shutil.copy(f'/content/{f}', f'{base}/{f}') for f in ['download_data.py','train_v2.py','gradio_app_v4.py']]; print('organized OK')

FileNotFoundError: [Errno 2] No such file or directory: '/content/__init__.py'

In [7]:
ase='/content/denoiser-rt/denoiser-rt'; os.makedirs(base+'/denoiser', exist_ok=True); os.makedirs(base+'/checkpoints', exist_ok=True); open(base+'/denoiser/__init__.py','w').close(); [shutil.copy(f'/content/{f}', f'{base}/denoiser/{f}') for f in ['stft.py','model.py']]; [shutil.copy(f'/content/{f}', f'{base}/{f}') for f in ['download_data.py','train_v2.py','gradio_app_v4.py']]; print('organized OK')

organized OK


In [10]:
!python /content/denoiser-rt/denoiser-rt/download_data.py

Baixando LibriSpeech dev-clean (~337MB, OpenSLR)...
+ wget -q -O /content/dev-clean.tar.gz https://www.openslr.org/resources/12/dev-clean.tar.gz
+ tar -xzf /content/dev-clean.tar.gz -C /content/dataset/clean_speech
+ rm -f /content/dev-clean.tar.gz
Fala real (LibriSpeech): 2703 arquivos .flac
  musica real ok: brahms (45.8s)
  musica real ok: nutcracker (119.9s)
  musica real ok: vibeace (61.5s)
  musica real ok: sweetwaltz (49.2s)
  musica real ok: pistachio (70.8s)
  musica real ok: trumpet (5.3s)
Musica real: 6 faixas
Babble sintetizado a partir de fala real: 60 clipes de 5s
Babble: 60 clipes

RESUMO DOS DADOS REAIS DISPONIVEIS
  fala (LibriSpeech dev-clean): 2703 arquivos
  musica real (librosa examples): 6 faixas
  babble (soma de fala real, tipo cocktail-party): 60 clipes

Obs.: ainda NAO ha ruido real de cozinha/restaurante do Kharina aqui.
Isso so entra quando voce subir seus .wav reais (o zip que corrompeu).


In [11]:
!DENOISER_STEPS=50 python /content/denoiser-rt/denoiser-rt/train_v2.py

Carregando fala real (LibriSpeech dev-clean)...
  265 clipes de fala
Carregando musica real...
  6 faixas de musica
Carregando babble (soma de vozes reais)...
  60 clipes de babble
device: cpu

Iniciando treino V4 (50 passos, batch 12, alvo limitado a [0,1])...

passo     0 | train 0.12443 | val 0.12447  <- salvo (melhor val)

Melhor val loss: 0.12447
Checkpoint final em: /content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt


In [12]:
import glob, os; [print(d, len(glob.glob(f'/content/dataset/{d}/**/*', recursive=True))) for d in os.listdir('/content/dataset')]

noise_babble 60
noise_music 6
clean_speech 2944
.ipynb_checkpoints 0
noise_kitchen 32


[None, None, None, None, None]

In [13]:
import shutil; shutil.copy('/content/train_v2.py', '/content/denoiser-rt/denoiser-rt/train_v2.py'); print('updated train_v2.py')

updated train_v2.py


In [ ]:
in_v2.py

In [ ]:
in_v2.py

In [15]:
!DENOISER_STEPS=1500 python /content/denoiser-rt/denoiser-rt/train_v2.py

Carregando fala real (LibriSpeech dev-clean)...
  265 clipes de fala
Carregando musica real...
  6 faixas de musica
Carregando babble (soma de vozes reais)...
  60 clipes de babble
Carregando ruido REAL do Kharina (cozinha/salao gravado no restaurante)...
  32 clipes de ruido real do Kharina
  -> ruido real do Kharina incluido no treino com peso maior (3x)
device: cpu
Partindo do checkpoint existente (fine-tuning).

Iniciando treino V4 (1500 passos, batch 12, alvo limitado a [0,1])...

passo     0 | train 0.11712 | val 0.12738  <- salvo (melhor val)
passo   100 | train 0.08522 | val 0.09316  <- salvo (melhor val)
passo   200 | train 0.07643 | val 0.06935  <- salvo (melhor val)
passo   300 | train 0.05802 | val 0.05222  <- salvo (melhor val)
passo   400 | train 0.06413 | val 0.06907
passo   500 | train 0.06997 | val 0.07174
passo   600 | train 0.05740 | val 0.06145
passo   700 | train 0.05443 | val 0.05792
passo   800 | train 0.06933 | val 0.08149
passo   900 | train 0.06844 | val 0.064

In [16]:
exec(open('/content/denoiser-rt/denoiser-rt/gradio_app_v4.py').read())

Pesos V4 (dados reais) carregados de: /content/denoiser-rt/denoiser-rt/checkpoints/masknet.pt
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://061d8c5b2991439f74.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
